In [6]:
# imports
import torch
print(torch.__version__)
import torchvision
import torchvision.datasets as datasets
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

import numpy as np
import pandas as pd
from sklearn.calibration import calibration_curve
import matplotlib.pyplot as plt


2.12.1


In [7]:
transform = transforms.ToTensor()
mnist_trainset = datasets.MNIST(root='./mnist_data', train=True, download=True, transform=transform)
mnist_testset = datasets.MNIST(root='./mnist_data', train=False, download=True, transform=transform)

In [8]:
train_loader = DataLoader(mnist_trainset, batch_size=128, shuffle=True)
test_loader = DataLoader(mnist_testset, batch_size=128, shuffle=True)

In [9]:
class MLP(nn.Module):
    def __init__(self, width):
        super().__init__()
        # turn square pictures into flat vectors 
        self.flatten = nn.Flatten()
        # 784 is the number of pixels and the width of the hidden layer is going to vary
        self.hidden = nn.Linear(784, width)
        # using basic ReLU() which is just max(0, z)
        self.relu = nn.ReLU()
        # 10 is the number of digits / categories so that's how many outputs we will have
        self.output = nn.Linear(width, 10)

    def forward(self, x):
        # turn these into long vectors of pixels
        x = self.flatten(x) 
        # multiply by W_1 matrix and add biases
        z = self.hidden(x)
        # apply the activation functions
        features = self.relu(z)        
        # multiply by W_2 matrix and add biases
        logits = self.output(features)
        # these will be logits
        return features, logits

In [10]:
model = MLP(width=128)

In [11]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.SGD(model.parameters(), lr=0.01, momentum=0.9)

In [12]:
losses = []
train_accs = []
test_accs = []
all_epochs_features = []

In [13]:
def train_epoch(return_features=False):
    # put the model in train mode
    model.train()
    # used to track loss so you can see if the training is working
    total_loss = 0
    epoch_features = []
    
    # mini-batch SGD
    for x, y in train_loader:
        optimizer.zero_grad()
        features, logits = model(x)
        epoch_features.append(features)
        loss = criterion(logits, y)
        # compute the gradient and store it in .grad attributes of each of the model parameters
        loss.backward()
        # optimizer can access attributes of model parameters and update the parameters
        optimizer.step()
        total_loss += loss.item()
    if return_features:
        return epoch_features, total_loss / len(train_loader)
    return total_loss / len(train_loader)

In [14]:
# evaluate the model's accuracy 
def evaluate_acc(model, loader):
    # put the model in test mode
    model.eval()
    
    correct = 0
    total = 0

    with torch.no_grad():

        for x,y in loader:
            # this calls forward with some wrappers that make it work better i guess
            _, logits = model(x)
            # model predicts the largest value to be the digit
            pred = logits.argmax(dim=1)
            # item makes it a number 
            correct += (pred == y).sum().item()
            total += y.size(0)

    return correct / total

In [15]:
for epoch in range(10):
    train_acc = evaluate_acc(model, train_loader)
    test_acc = evaluate_acc(model, test_loader)

    # loss = train_epoch()

    epoch_features, loss = train_epoch(return_features=True)
    all_epochs_features.append(epoch_features)

    losses.append(loss)
    train_accs.append(train_acc)
    test_accs.append(test_acc)
    
    print(
        epoch,
        loss,
        train_acc,
        test_acc
    )

0 0.6231239292540276 0.09801666666666667 0.0939
1 0.2900749397938694 0.9084166666666667 0.9119
2 0.23749104135834587 0.9291333333333334 0.9313
3 0.20186843939903956 0.9401166666666667 0.9408
4 0.17497092216952778 0.9490833333333333 0.9473
5 0.15479794017541637 0.95425 0.9522
6 0.13953939178732158 0.9601833333333334 0.9576
7 0.12615962188317578 0.9645166666666667 0.9612
8 0.11463893861023348 0.9677166666666667 0.9631
9 0.10589514864183693 0.9701166666666666 0.966


In [16]:
big_matrices = [torch.cat(epoch_list, dim=0) for epoch_list in all_epochs_features]

In [17]:
epoch_1_phi = big_matrices[1]

notes to self: 
- I'd like to graph GAD on some of these epoch graphs using the phi matrix as the design, possibly ordering columns by sum of W2 rows? 
- I could try to graph effective rank and/or singular values distribution like joseph did.
- I'd like to experiment with splicing in CK descent to save training time / replicating that result from Pacific Northwest
- I'd like to plot the difference in coefficients of CK best regression solution vs coefficients found by SGD (include bias so it's a fair comparison)